In [1]:
import numpy as np 
import tensorflow as tf 
import time 
import os 

2023-07-11 22:29:32.661944: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-07-11 22:29:32.924475: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-07-11 22:29:32.926731: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-11 22:29:33.991516: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [5]:
path= tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

text= open(path, "rb").read().decode("utf-8")

print("The length of the text is", len(text))

#Lets take a look at the first 149 elements 
print(text[:149])

unique_chars= sorted(set(text))
print("The number of unique characters are: ", len(unique_chars))

The length of the text is 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?


The number of unique characters are:  65


### Vectorize the text 
#### Before training, we need to convert the strings into a numerical ID which can be done using tf.keras.layers.StringLookup

In [7]:
chars_to_id= tf.keras.layers.StringLookup(vocabulary= list(unique_chars), mask_token= None)

#This line of code converts the tokens to character ID. 
ids= chars_to_id(unique_chars)
ids

<tf.Tensor: shape=(65,), dtype=int64, numpy=
array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51,
       52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65])>

#### We can revert the ids to strings using invert = True which returns the strings in RaggedTensor form which can be joined using tf.reduce_join as follows: 

In [9]:
id_to_chars= tf.keras.layers.StringLookup(vocabulary= chars_to_id.get_vocabulary(), invert= True, mask_token= None)
chars= id_to_chars(ids)

print("The returned strings in ragged tensor form", chars)

strings= tf.strings.reduce_join(chars, axis=-1).numpy()     
print("The strings after joining are  ", strings)

#lets define a function for turning the ids to strings: 
def id_to_text(ids): 
    return tf.strings.reduce_join(id_to_chars(ids), axis=-1)

The returned strings in ragged tensor form tf.Tensor(
[b'\n' b' ' b'!' b'$' b'&' b"'" b',' b'-' b'.' b'3' b':' b';' b'?' b'A'
 b'B' b'C' b'D' b'E' b'F' b'G' b'H' b'I' b'J' b'K' b'L' b'M' b'N' b'O'
 b'P' b'Q' b'R' b'S' b'T' b'U' b'V' b'W' b'X' b'Y' b'Z' b'a' b'b' b'c'
 b'd' b'e' b'f' b'g' b'h' b'i' b'j' b'k' b'l' b'm' b'n' b'o' b'p' b'q'
 b'r' b's' b't' b'u' b'v' b'w' b'x' b'y' b'z'], shape=(65,), dtype=string)
The strings after joining are   b"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
